# Chapter 2 · Function calling in depth

Function calling, layer by layer, with the agent of Northside Veterinary Clinic: the exact text the model receives, the special tokens of its calls, two parsers, several calls per step and how to make the model use a tool by writing the start of its answer. Two models are used, Qwen3-4B-Instruct-2507 and Qwen3-1.7B.

**Before you start, turn on the GPU:** menu *Runtime* → *Change runtime type* → *T4 GPU* → *Save*.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Load the model and the clinic

The clinic and its tools are those of chapter 1's toolset 4, with one more: `price`.

In [ ]:
!pip install -q transformers==5.19.0 mcp==2.3.0  # the versions the book was checked with

In [ ]:
"""Chapter 2: function calling in depth. What goes into the model, what comes out,
how the calls are read, several calls per step and forcing a tool."""
import copy
import json
import re
import time
import unicodedata
import urllib.request
from datetime import date
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

GPU_MODEL = "Qwen/Qwen3-4B-Instruct-2507"  # with Colab's free T4 GPU
CPU_MODEL = "Qwen/Qwen3-1.7B"              # without a GPU: smaller, and a worse agent
MODEL = None                               # None: choose according to the hardware
DATA = Path("data")
REPOSITORY = "https://raw.githubusercontent.com/HernanDiaz/ai-tools-mcp-skills/main/en/"
TODAY = date(2026, 10, 20)                 # the date of the clinic's data


def load(name=None):
    """Loads the model and its tokenizer: on the GPU, in float16; if not, on the CPU."""
    gpu = torch.cuda.is_available()
    if name is None:
        name = GPU_MODEL if gpu else CPU_MODEL
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(
        name, dtype=torch.float16 if gpu else torch.float32)
    model.to("cuda" if gpu else "cpu").eval()
    print(f"Model {name} on {'cuda' if gpu else 'cpu'}")
    return tok, model


def download(name):
    """A JSON file from the data folder; in Colab, it is downloaded."""
    path = DATA / name
    if not path.exists():
        DATA.mkdir(exist_ok=True)
        urllib.request.urlretrieve(REPOSITORY + "data/" + name, path)
    return json.loads(path.read_text(encoding="utf-8"))

In [ ]:
tok, model = load(MODEL)

In [ ]:
def reset():
    """Puts the clinic back as it was: the original schedule and no notices."""
    global APPOINTMENTS, NOTICES
    APPOINTMENTS, NOTICES = copy.deepcopy(APPOINTMENTS0), []


def strip_accents(text):
    """Lowercase and without diacritics, to compare names written in any form."""
    text = unicodedata.normalize("NFKD", str(text).lower())
    return "".join(c for c in text if not unicodedata.combining(c)).strip()


def book_appointment(pet, vet, date, time, reason):
    """Books a free slot in the schedule; returns the appointment or the problem."""
    if pet not in PETS:
        return {"error": f"There is no pet {pet}."}
    if vet not in VETS:
        return {"error": f"There is no vet {vet}."}
    if any(c["date"] == date and c["time"] == time and c["vet"] == vet
           and c["status"] != "canceled" for c in APPOINTMENTS):
        return {"error": f"{VETS[vet]} already has an appointment on {date} "
                         f"at {time}."}
    appointment = {"id": f"AP-{len(APPOINTMENTS0) + len(APPOINTMENTS) + 1:04d}",
                   "date": date, "time": time, "vet": vet, "pet": pet,
                   "reason": reason, "status": "scheduled"}
    APPOINTMENTS.append(appointment)
    return appointment


def cancel_appointment(appointment):
    """Cancels a scheduled appointment."""
    for c in APPOINTMENTS:
        if c["id"] == appointment:
            if c["status"] != "scheduled":
                return {"error": f"Appointment {appointment} is \"{c['status']}\"."}
            c["status"] = "canceled"
            return {"canceled": appointment, "date": c["date"], "time": c["time"]}
    return {"error": f"There is no appointment {appointment}."}


def notify(owner, text):
    """Sends a notice to the owner (here it is stored in a list)."""
    if owner not in OWNERS:
        return {"error": f"There is no client {owner}."}
    NOTICES.append({"owner": owner, "text": text})
    return {"sent": True, "to": OWNERS[owner]["name"],
            "via": OWNERS[owner]["notify_by"]}


def age(birth_date):
    """Years completed on the date of the data."""
    n = date.fromisoformat(birth_date)
    return TODAY.year - n.year - ((TODAY.month, TODAY.day) < (n.month, n.day))


def pet_summary(pid):
    """The essentials of a pet, without the full medical history."""
    m = PETS[pid]
    return {"id": pid, "name": m["name"], "species": m["species"], "breed": m["breed"],
            "age_years": age(m["birth_date"]), "current_weight_kg": m["weight_kg"],
            "allergies": m["allergies"] or "none known",
            "owner": {"id": m["owner"], "name": OWNERS[m["owner"]]["name"]}}


def vaccines_with_status(pid):
    """Each vaccine with its next date and whether it is already overdue."""
    return [{"vaccine": v["vaccine"], "next": v["next"],
             "status": "overdue" if v["next"] < TODAY.isoformat() else "up to date"}
            for v in VACCINES[pid]]


def client_record_3(client_id):
    """The client's record, summarized, with the total they owe."""
    if client_id not in OWNERS:
        return {"error": f"There is no client {client_id}. Look them up by name with "
                         "search(table='owners', field='name', value=...)."}
    unpaid = [f for f in INVOICES if f["owner"] == client_id and not f["paid"]]
    return {"id": client_id, "name": OWNERS[client_id]["name"],
            "notify_by": OWNERS[client_id]["notify_by"],
            "pets": [{"id": p, "name": d["name"], "species": d["species"]}
                     for p, d in PETS.items() if d["owner"] == client_id],
            "unpaid_invoices": [{"id": f["id"], "date": f["date"], "total": f["total"]}
                                for f in unpaid],
            "total_unpaid": round(sum(f["total"] for f in unpaid), 2)}


def schedule_3(date):
    """The day's appointments, one per line, and each vet's free times."""
    if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", str(date)):
        return {"error": f"The date \"{date}\" is not valid; use YYYY-MM-DD."}
    appts = [c for c in APPOINTMENTS if c["date"] == date and c["status"] != "canceled"]
    return {"date": date, "appointments": [f"{c['id']} {c['time']} {c['vet']} "
                                           f"{PETS[c['pet']]['name']} ({c['pet']})"
                                           for c in appts],
            "vets": VETS,
            "free_times": {v: [t for t in TIMES if not any(
                c["time"] == t and c["vet"] == v for c in appts)]
                for v in VETS}}


def locate(pet, owner=""):
    """The pet's id from its name (and the owner's, if there are several)."""
    ids = [m for m, d in PETS.items() if strip_accents(pet) == strip_accents(d["name"])
           and strip_accents(owner) in strip_accents(OWNERS[d["owner"]]["name"])]
    if not ids:
        return None, {"error": f"There is no pet called \"{pet}\""
                               + (f" owned by {owner}" if owner else "")
                               + " at the clinic."}
    if len(ids) > 1:
        return None, {"error": f"There are {len(ids)} pets called \"{pet}\"; give "
                               "the owner's name.",
                      "pets": [pet_summary(m) for m in ids]}
    return ids[0], None


def vet_of(name):
    """The vet's id from their name or their id."""
    for v, n in VETS.items():
        if strip_accents(name) in (strip_accents(v), strip_accents(n)) or \
                strip_accents(name) in strip_accents(n):
            return v
    return None


def find_pet(name, owner=""):
    """A pet's details, looked up by name."""
    pid, error = locate(name, owner)
    return error or pet_summary(pid)


def pets_of(owner):
    """The pets of the clients whose name contains the one given."""
    ids = [d for d, x in OWNERS.items()
           if strip_accents(owner) in strip_accents(x["name"])]
    if not ids:
        return {"error": f"There is no client called \"{owner}\"."}
    return {"clients": [{"name": OWNERS[d]["name"], "pets": [
        {"name": x["name"], "species": x["species"], "breed": x["breed"]}
        for m, x in PETS.items() if x["owner"] == d]} for d in ids]}


def vaccines(pet, owner=""):
    """A pet's vaccines, each with its next date and status."""
    pid, error = locate(pet, owner)
    return error or {"pet": PETS[pid]["name"], "today": TODAY.isoformat(),
                     "vaccines": vaccines_with_status(pid)}


def history(pet, last=3, owner=""):
    """A pet's last visits."""
    pid, error = locate(pet, owner)
    if error:
        return error
    return {"pet": PETS[pid]["name"], "visits": [
        {k: v[k] for k in ("date", "reason", "treatment", "weight_kg")}
        for v in HISTORY[pid][-int(last):]]}


def dose(medicine, pet, owner=""):
    """The dose of a medicine for a pet, with allergy and species warnings."""
    med = PHARMACY.get(strip_accents(medicine))
    if med is None:
        return {"error": f"We don't have \"{medicine}\". Medicines: "
                         + ", ".join(PHARMACY) + "."}
    pid, error = locate(pet, owner)
    if error:
        return error
    m = PETS[pid]
    warnings = []
    if strip_accents(medicine) in [strip_accents(a) for a in m["allergies"]]:
        warnings.append(f"{m['name']} is allergic to {medicine}: do not prescribe.")
    if m["species"] not in med["species"]:
        warnings.append(f"{medicine} is not indicated for {m['species']}s.")
    return {"medicine": strip_accents(medicine), "pet": m["name"],
            "weight_kg": m["weight_kg"], "dose_mg_per_kg": med["dose_mg_kg"],
            "dose_mg_per_dose": round(med["dose_mg_kg"] * m["weight_kg"], 2),
            "stock": med["stock"],
            "restock": med.get("restock", "no date") if not med["stock"] else None,
            "warnings": warnings or "none"}


def free_slots(date, vet=""):
    """The free times of a day, for one vet or for all of them."""
    r = schedule_3(date)
    if "error" in r:
        return r
    vets = [vet_of(vet)] if vet else list(VETS)
    if vets == [None]:
        return {"error": f"There is no vet \"{vet}\". They are: "
                         + ", ".join(VETS.values()) + "."}
    return {"date": date, "free_times": {VETS[v]: r["free_times"][v] for v in vets}}


def appointments_of(pet, owner=""):
    """A pet's appointments, past and upcoming, with their id."""
    pid, error = locate(pet, owner)
    return error or {"pet": PETS[pid]["name"], "today": TODAY.isoformat(),
                     "appointments": [
        {"id": c["id"], "date": c["date"], "time": c["time"], "reason": c["reason"],
         "vet": VETS[c["vet"]], "status": c["status"]}
        for c in APPOINTMENTS if c["pet"] == pid]}


def accounts(owner):
    """The unpaid invoices of the clients whose name contains the one given."""
    ids = [d for d, x in OWNERS.items()
           if strip_accents(owner) in strip_accents(x["name"])]
    if not ids:
        return {"error": f"There is no client called \"{owner}\"."}
    return {"clients": [{k: v for k, v in client_record_3(d).items() if k != "id"}
                        for d in ids]}


def book(pet, vet, date, time, reason, owner=""):
    """Books an appointment from the names of the pet and the vet."""
    pid, error = locate(pet, owner)
    if error:
        return error
    v = vet_of(vet)
    if v is None:
        return {"error": f"There is no vet \"{vet}\". They are: "
                         + ", ".join(VETS.values()) + "."}
    r = book_appointment(pid, v, date, time, reason)
    if "error" in r:
        return r
    return {"booked": r["id"], "pet": PETS[pid]["name"],
            "vet": VETS[v], "date": date, "time": time}


def notify_owner_of(pet, text, owner=""):
    """Sends a notice to the owner of the pet."""
    pid, error = locate(pet, owner)
    return error or notify(PETS[pid]["owner"], text)


def words(text):
    """The roots (first seven letters) of the content words of a text."""
    return {w[:7] for w in re.findall(r"\w{4,}", strip_accents(text))}


def price(service):
    """Price of a service of the clinic or of a medicine of the pharmacy: the ones
    that share most words with what is asked for."""
    prices = {**PRICES, **{k: d["price"] for k, d in PHARMACY.items()}}
    common = {k: len(words(service) & words(k)) for k in prices}
    best = max(common.values())
    if best == 0:
        return {"error": f"There is no price for \"{service}\". Services: "
                         + ", ".join(PRICES) + "; and the medicines of the pharmacy."}
    return {"prices_usd": {k: prices[k] for k, c in common.items() if c == best}}


FUNCTIONS = {"find_pet": find_pet, "pets_of": pets_of,
             "vaccines": vaccines, "history": history, "dose": dose,
             "free_slots": free_slots, "appointments_of": appointments_of,
             "accounts": accounts, "book_appointment": book,
             "cancel_appointment": cancel_appointment, "notify_owner": notify_owner_of,
             "price": price}

In [ ]:
def tool(name, description, parameters):
    """Schema of a tool; the parameters without a default value are required."""
    return {"type": "function", "function": {
        "name": name, "description": description,
        "parameters": {"type": "object",
                       "required": [p for p, (_, req) in parameters.items() if req],
                       "properties": {p: {"type": "string", "description": d}
                                      for p, (d, _) in parameters.items()}}}}


PET = ("Name of the pet.", True)


OWNER = ("Owner's name; only if several pets have that name.", False)


SCHEMAS_TASKS = [
    tool("find_pet", "Details of a pet looked up by its name: species, age, current "
         "weight, allergies and owner.", {"name": PET, "owner": OWNER}),
    tool("pets_of", "The pets of a client, looked up by name.",
         {"owner": ("Owner's name.", True)}),
    tool("vaccines", "Vaccines of a pet: the next date of each one and whether it is "
         "overdue.", {"pet": PET, "owner": OWNER}),
    tool("history", "The last visits of a pet: date, reason, treatment and weight.",
         {"pet": PET, "last": ("How many visits (3 by default).", False),
          "owner": OWNER}),
    tool("dose", "Dose of a medicine for a pet according to its current weight, with "
         "allergy or species warnings and the stock.",
         {"medicine": ("Name of the medicine.", True), "pet": PET, "owner": OWNER}),
    tool("free_slots", "Free times in the schedule on a day, for one vet or for all "
         "of them.", {"date": ("YYYY-MM-DD.", True),
                      "vet": ("Name of the vet; empty for all.", False)}),
    tool("appointments_of", "Appointments of a pet (past and upcoming), with their id.",
         {"pet": PET, "owner": OWNER}),
    tool("accounts", "Unpaid invoices of a client, looked up by name, and the total "
         "they owe.", {"owner": ("Owner's name.", True)}),
    tool("book_appointment", "Books an appointment at a free time.",
         {"pet": PET, "vet": ("Name of the vet.", True),
          "date": ("YYYY-MM-DD.", True),
          "time": ("HH:MM, on the hour or half past.", True),
          "reason": ("Reason for the appointment.", True), "owner": OWNER}),
    tool("cancel_appointment", "Cancels a scheduled appointment.",
         {"appointment": ("Id of the appointment, such as AP-0026 (given by "
                          "appointments_of).", True)}),
    tool("notify_owner", "Sends a notice to the owner of a pet, by email or SMS as "
         "they prefer.", {"pet": PET, "text": ("Text of the notice, in English.", True),
                          "owner": OWNER}),
]


SCHEMAS = SCHEMAS_TASKS + [
    tool("price", "Price in dollars of a service of the clinic (consultation, "
         "vaccines, tests…) or of a medicine.",
         {"service": ("Service or medicine.", True)})]


BASE = (
    "You are the assistant of the staff of Northside Veterinary Clinic, a veterinary "
    "clinic in Portland, Oregon. Today is Tuesday 2026-10-20. Do the task step by step "
    "using the tools. Never invent data that a tool can give you, and never say that "
    "an action was done unless a tool confirmed it. ")


FINAL = "When the task is finished, answer in English, in one or two sentences."


SYSTEM_ONE = BASE + (
    "Call only one tool at a time, and before each tool call write one short line "
    "starting with 'Thought:' that says what you already know and what you need "
    "next. ") + FINAL


SYSTEM_SEVERAL = BASE + (
    "Before calling tools, write one short line starting with 'Thought:' that says "
    "what you already know and what you need next. If several calls do not depend on "
    "each other, request them all in the same turn. ") + FINAL

## 2. Layer 1: what goes in

The exact text the model receives, and how many tokens each part takes.

In [ ]:
def template(messages, schemas):
    """The exact text the model receives, with the tools included."""
    return tok.apply_chat_template(messages, tools=schemas, tokenize=False,
                                   add_generation_prompt=True, enable_thinking=False)


def count(text):
    """Tokens of a text, without special tokens added."""
    return len(tok(text, add_special_tokens=False)["input_ids"])


def input_parts(task, system=SYSTEM_ONE, schemas=SCHEMAS):
    """How many tokens each part of the input text takes in the first step."""
    text = template([{"role": "system", "content": system},
                     {"role": "user", "content": task}], schemas)
    start, end = text.rindex("<tools>"), text.rindex("</tools>") + len("</tools>")
    tools = text[start:end]
    instructions = (text[text.index("# Tools"):start]
                    + text[end:text.index("<|im_end|>")])
    return {"total": count(text), "system": count(system), "task": count(task),
            "tools": count(tools), "call_instructions": count(instructions)}

In [ ]:
MONTHS = ["January", "February", "March", "April", "May", "June", "July", "August",
          "September", "October", "November", "December"]


def says(answer, *forms):
    """Does any of those forms appear in the answer?"""
    return any(re.search(f, answer or "", re.I) for f in forms)


def date_in(answer, iso):
    """Does the date appear, written as 2027-08-22, 8/22/2027, August 22, 2027, Aug 22
    or 22 August?"""
    y, m, d = iso.split("-")
    month = MONTHS[int(m) - 1]
    name = rf"(?:{month}|{month[:3]}\.?)"
    return says(answer, iso, rf"\b0?{int(m)}/0?{int(d)}/{y}\b",
                rf"\b{name} 0?{int(d)}(?:st|nd|rd|th)?\b(?:,? {y})?",
                rf"\b{int(d)}(?:st|nd|rd|th)? (?:of )?{name}\b")


def time_in(answer, hhmm):
    """Does the time appear, as 16:30, 4:30 p.m., 4:30 pm or 4:30pm (4 p.m. for 16:00)?
    In a list such as "4:00, 4:30 and 5:30 p.m.", a bare 4:30 counts too."""
    h, m = map(int, hhmm.split(":"))
    h12, half = (h - 1) % 12 + 1, "p" if h >= 12 else "a"
    other = "a" if half == "p" else "p"
    forms = [rf"(?<![\d:]){h:02d}:{m:02d}(?!\d)",
             rf"(?<![\d:]){h12}:{m:02d}(?!\d)(?!\s*{other}\.?\s?m\b)"]
    if m == 0:
        forms.append(rf"(?<![\d:]){h12}\s?{half}\.?\s?m\b")
    if (h, m) == (12, 0):
        forms.append(r"\bnoon\b")
    return says(answer, *forms)


def new_appointments():
    """The appointments created during the task."""
    return [c for c in APPOINTMENTS if c["id"] not in {x["id"] for x in APPOINTMENTS0}]


def first_free(date, vet):
    """The vet's first free time on that day, in the original schedule."""
    taken = {c["time"] for c in APPOINTMENTS0 if c["date"] == date and c["vet"] == vet}
    return next(t for t in TIMES if t not in taken)


def owes(name):
    """What a client owes in unpaid invoices."""
    return round(sum(f["total"] for f in INVOICES if not f["paid"]
                     and OWNERS[f["owner"]]["name"] == name), 2)


def amount_in(answer, value):
    """Does the amount appear, as 136, 136.00, $136 or 136 dollars?"""
    whole = f"{value:.2f}".replace(".00", "")
    return says(answer, rf"\b{re.escape(whole)}(\.00?)?\b")


def notice_to(name, *forms):
    """Was a notice sent to that client, containing all those forms?"""
    return any(OWNERS[a["owner"]]["name"] == name
               and all(re.search(f, a["text"], re.I) for f in forms) for a in NOTICES)


ANY_DATE = (r"\b20\d\d-\d\d-\d\d\b", r"\b\d{1,2}/\d{1,2}/20\d\d\b",
            rf"\b(?:{'|'.join(m[:3] for m in MONTHS)})[a-z]*\.? \d{{1,2}}\b")


TASKS = [
    ("rabies_vaccine", "When is Lola, Martha Fisher's dog, due for her next rabies "
     "vaccine?",
     lambda r: date_in(r, "2027-08-22")),
    ("dose", "How many milligrams of meloxicam should Max get per dose?",
     lambda r: says(r, r"2\.07", r"2\.1\b")),
    ("allergy", "They're going to prescribe amoxicillin for Simba. Is there any "
     "problem?",
     lambda r: says(r, r"allerg")),
    ("book", "Book a checkup for Toby with Dr. Paul Meyer on Thursday the 22nd, at the "
     "first free time.",
     lambda r: [(c["pet"], c["vet"], c["date"], c["time"]) for c in new_appointments()]
     == [("P-002", "V2", "2026-10-22", first_free("2026-10-22", "V2"))]),
    ("cancel", "Cancel Rocky's appointment tomorrow.",
     lambda r: [c["id"] for c in APPOINTMENTS if c["status"] == "canceled"]
     == ["AP-0060"]),
    ("overdue_notice", "Let Toby's owner know about his overdue vaccines.",
     lambda r: notice_to("Peter Adams", "rabies", "DHPP", "leptospirosis")),
    ("debt", "How much does Oscar Price owe in unpaid invoices?",
     lambda r: amount_in(r, owes("Oscar Price"))),
    ("marthas_cat", "Martha Fisher is calling about her cat. Is any of her vaccines "
     "overdue?",
     lambda r: says(r, "FVRCP")
     and says(r, "overdue|expired|out of date|past due|lapsed|behind")),
    ("weight", "How much did Zeus weigh at his last visit, and how much has it changed "
     "since the one before?",
     lambda r: says(r, r"32\.2") and says(r, r"0\.6")
     and not says(r, r"(last|most recent) visit[^.,]{0,30}(2025|31\.6)",
                  r"31\.6 kg (at|in|during) (his|the) (last|most recent)")),
    ("last_visit", "Why did Willow come in last time, and what treatment did she get?",
     lambda r: says(r, "wound") and says(r, "amoxicillin")),
    ("free_slots", "What times is Dr. Anna Sutton free on Wednesday the 21st in the "
     "afternoon?",
     lambda r: all(time_in(r, t) for t in ("16:00", "16:30", "17:30", "18:00", "18:30"))
     and not time_in(r, "17:00")),
    ("does_not_exist", "When is Buster due for his rabies vaccine?",
     lambda r: says(r, r"\bno (pet|record|patient|animal|one|such|information)",
                    r"\bthere(?: is|'s) no\b", r"\bunable to (find|locate)",
                    r"(n't|\bnot|cannot) (exist|find|have|appear|see|locate|found|"
                    r"registered|listed|on file|in our)")
     and not says(r, *ANY_DATE)),
]


def load_data():
    """Loads the clinic's data and resets it."""
    global PETS, OWNERS, HISTORY, VACCINES, APPOINTMENTS0, PHARMACY, INVOICES
    global VETS, TIMES, PRICES, ANSWERS
    PETS, OWNERS = download("pets.json"), download("owners.json")
    HISTORY, VACCINES = download("history.json"), download("vaccines.json")
    APPOINTMENTS0, PHARMACY = download("appointments.json"), download("pharmacy.json")
    INVOICES, VETS = download("invoices.json"), download("vets.json")
    PRICES = download("prices.json")
    TIMES = [f"{h:02d}:{m:02d}" for h in range(9, 14) for m in (0, 30)] + \
            [f"{h:02d}:{m:02d}" for h in range(16, 19) for m in (0, 30)]
    ANSWERS = []
    reset()

In [ ]:
load_data()
task = ("How much do Oscar Price, Paula Ingram and Laura Green owe in total, "
        "the three of them together?")
text = template([{"role": "system", "content": SYSTEM_ONE},
                 {"role": "user", "content": task}], SCHEMAS[:2])
print(text)
print(input_parts(task))

## 3. Layer 2: what comes out

`<tool_call>` is a single token of the vocabulary.

In [ ]:
print(tok("<tool_call>", add_special_tokens=False)["input_ids"])

In [ ]:
def generate(messages, schemas, prefix="", max_tokens=400):
    """One answer of the model. With a prefix, the text of the answer starts with it."""
    text = template(messages, schemas) + prefix
    inputs = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=max_tokens,
                                do_sample=False, temperature=None, top_p=None,
                                top_k=None, pad_token_id=tok.eos_token_id)
    new = output[0, inputs["input_ids"].shape[1]:]
    answer = prefix + tok.decode(new, skip_special_tokens=True)
    return answer.strip(), inputs["input_ids"].shape[1], len(new)

In [ ]:
answer, n_in, n_out = generate([{"role": "system", "content": SYSTEM_SEVERAL},
                                {"role": "user", "content": task}], SCHEMAS)
print(n_in, "input tokens,", n_out, "output tokens")
print(answer)

## 4. Layer 3: reading the calls

In [ ]:
def read_naive(answer):
    """The simplest parser: the JSON after the first <tool_call>."""
    if "<tool_call>" not in answer:
        return []
    block = answer.split("<tool_call>")[1].split("</tool_call>")[0]
    try:
        call = json.loads(block)
        return [{"name": call["name"], "arguments": call.get("arguments", {})}]
    except (ValueError, KeyError, TypeError):
        return [{"name": None, "arguments": {}}]


def read_calls(answer):
    """All the calls, each between <tool_call> and </tool_call> (chapter 1's)."""
    calls = []
    for block in re.findall(r"<tool_call>(.*?)</tool_call>", answer, re.S):
        try:
            call = json.loads(block)
            calls.append({"name": call["name"],
                          "arguments": call.get("arguments", {})})
        except (ValueError, KeyError, TypeError):
            calls.append({"name": None, "arguments": {}})
    return calls


PARSERS = {"naive": read_naive, "full": read_calls}

In [ ]:
for name, read in PARSERS.items():
    print(name, read(answer))

## 5. The agent, with several calls per step and with the first one forced

In [ ]:
def execute(call):
    """Executes a call if the tool exists and the arguments fit."""
    function = FUNCTIONS.get(call["name"])
    if function is None:
        return {"error": f"There is no tool {call['name']}."}
    try:
        return function(**call["arguments"])
    except (TypeError, ValueError, KeyError) as e:
        return {"error": f"Invalid arguments: {e}"}


def agent(task, system=SYSTEM_ONE, several=False, force=False, read=read_calls,
          max_steps=8, show=False):
    """Chapter 1's loop. With several, it runs all the calls of a step; with force,
    the first step starts with <tool_call> and has to call a tool. It stores each
    answer as is in ANSWERS, to test the parsers afterwards."""
    messages = [{"role": "system", "content": system},
                {"role": "user", "content": task}]
    trace = []
    for step in range(1, max_steps + 1):
        prefix = "<tool_call>\n" if force and step == 1 else ""
        answer, n_in, n_out = generate(messages, SCHEMAS, prefix)
        ANSWERS.append(answer)
        calls = read(answer)
        if not several:
            calls = calls[:1]
        thought = answer.split("<tool_call>")[0].strip()
        trace.append({"step": step, "input_tokens": n_in, "output_tokens": n_out,
                      "thought": thought, "calls": []})
        if not calls:
            if show:
                print(f"[{step}] Answer: {answer}")
            return answer, trace
        if show and thought:
            print(f"[{step}] {thought}")
        messages.append({"role": "assistant", "content": thought, "tool_calls": [
            {"type": "function", "function": c} for c in calls if c["name"]]})
        for call in calls:
            result = execute(call)
            trace[-1]["calls"].append({**call, "result": result})
            messages.append({"role": "tool", "name": call["name"] or "?",
                             "content": json.dumps(result, ensure_ascii=False)})
            if show:
                args = json.dumps(call["arguments"], ensure_ascii=False)
                print(f"[{step}] {call['name']}({args})")
    return None, trace

In [ ]:
reset()
agent(task, system=SYSTEM_SEVERAL, several=True, show=True)

The parallel tasks, the price questions and the functions that solve and measure them:

In [ ]:
def overdue(name):
    """The overdue vaccines of a pet, by its name."""
    pid = next(m for m, d in PETS.items() if d["name"] == name)
    return [v["vaccine"] for v in vaccines_with_status(pid) if v["status"] == "overdue"]


def last_visit(name):
    """The date of a pet's last visit, by its name."""
    pid = next(m for m, d in PETS.items() if d["name"] == name)
    return HISTORY[pid][-1]["date"]


def dose_of(medicine, name):
    """The dose of a medicine for a pet, by its name."""
    m = next(d for d in PETS.values() if d["name"] == name)
    return PHARMACY[medicine]["dose_mg_kg"] * m["weight_kg"]


def number_in(answer, value):
    """Does the figure appear, with one or two decimals?"""
    return says(answer, *{re.escape(f"{value:.2f}"), re.escape(f"{value:.1f}") + r"\b",
                          rf"\b{re.escape(f'{value:g}')}\b"})


# Tasks with several lookups that are independent of each other
PARALLEL = [
    ("three_debts", "How much do Oscar Price, Paula Ingram and Laura Green owe in "
     "total, the three of them together?",
     lambda r: amount_in(r, owes("Oscar Price") + owes("Paula Ingram")
                         + owes("Laura Green"))),
    ("three_doses", "How many milligrams of meloxicam per dose should Max, Zeus and "
     "Rufus get?",
     lambda r: all(number_in(r, round(dose_of("meloxicam", n), 2))
                   for n in ("Max", "Zeus", "Rufus"))),
    ("last_visits", "When did Rocky, Willow and Tiger last come in?",
     lambda r: all(date_in(r, last_visit(n)) for n in ("Rocky", "Willow", "Tiger"))),
    ("three_overdue", "Do Bruno, Nala or Salem have any overdue vaccines? Tell me "
     "which.",
     lambda r: all(says(r, v.split()[0]) for n in ("Bruno", "Nala", "Salem")
                   for v in overdue(n))),
    ("peters_pets", "Which vaccines are overdue for Peter Adams's pets?",
     lambda r: all(says(r, v) for v in overdue("Toby") + overdue("Snowball"))),
    ("notify_two", "Let the owners of Oliver and Salem know that their FVRCP vaccine "
     "is overdue.",
     lambda r: notice_to("Claire Miller", "FVRCP")
     and notice_to("Rose Vance", "FVRCP")),
    ("two_vets_free", "What times are Dr. Paul Meyer and Dr. Lucy Fenton free on "
     "Friday the 23rd in the morning?",
     lambda r: all(time_in(r, t) for v in ("V2", "V3")
                   for t in morning_free("2026-10-23", v))),
    ("tobys_price", "How much would it cost to give Toby all his overdue vaccines?",
     lambda r: amount_in(r, sum(PRICES[v + " vaccine"] for v in overdue("Toby")))),
]


def morning_free(date, vet):
    """The vet's free morning times on that day, in the original schedule."""
    taken = {c["time"] for c in APPOINTMENTS0 if c["date"] == date and c["vet"] == vet
             and c["status"] != "canceled"}
    return [t for t in TIMES if t < "14:00" and t not in taken]


# Price questions, which the model could answer from memory
OUT_OF_STOCK = (r"out of stock", r"none left", r"not available", r"unavailable",
                r"\b0 units", r"zero units", r"(don't|do not|doesn't) have (it|any)",
                r"\bno (stock|units)\b", r"\b(ran|run) out\b", r"\b0\b")


PRICE_QUESTIONS = [
    ("cleaning", "How much is a dental cleaning?", lambda r: amount_in(r, 120)),
    ("fvrcp", "How much do we charge for the FVRCP vaccine?",
     lambda r: amount_in(r, 36)),
    ("meloxicam", "How much is meloxicam?", lambda r: number_in(r, 18.5)),
    ("blood_test", "What's the price of a blood test?", lambda r: amount_in(r, 65)),
    ("consultation", "How much is a consultation?", lambda r: amount_in(r, 35)),
    ("myxomatosis", "How much does it cost to vaccinate a bunny against myxomatosis?",
     lambda r: amount_in(r, 25)),
    ("deworming", "A client asks how much deworming their dog would cost.",
     lambda r: amount_in(r, 15)),
    ("doxycycline", "How much is doxycycline, and do we have it right now?",
     lambda r: number_in(r, 14.2) and says(r, *OUT_OF_STOCK)),
]


def solve(tasks, max_steps=8, **options):
    """Solves each task with the clinic reset and records the result."""
    rows = []
    for key, task, check in tasks:
        reset()
        start = time.time()
        answer, trace = agent(task, max_steps=max_steps, **options)
        rows.append({"task": key, "ok": bool(check(answer)), "answer": answer,
                     "steps": len(trace), "trace": trace,
                     "calls": [(c["name"], c["arguments"])
                               for s in trace for c in s["calls"]],
                     "input_tokens": sum(s["input_tokens"] for s in trace),
                     "seconds": round(time.time() - start, 1), "notices": NOTICES})
    return rows


def summary(rows):
    """The totals of a variant over all its tasks."""
    return {"ok": sum(r["ok"] for r in rows),
            "finished": sum(r["answer"] is not None for r in rows),
            "steps": sum(r["steps"] for r in rows),
            "calls": sum(len(r["calls"]) for r in rows),
            "no_tools": sum(not r["calls"] for r in rows),
            "input_tokens": sum(r["input_tokens"] for r in rows),
            "seconds": round(sum(r["seconds"] for r in rows), 1)}


def count_calls(answers):
    """For each parser, the valid calls it finds in the answers, against the ones the
    model started to write (the <tool_call> tags)."""
    written = sum(a.count("<tool_call>") for a in answers)
    return {"written": written, **{name: sum(
        sum(c["name"] in FUNCTIONS and isinstance(c["arguments"], dict)
            for c in read(a))
        for a in answers) for name, read in PARSERS.items()}}

In [ ]:
results = {}
for name, tasks, options in [
        ("Parallel, one at a time", PARALLEL, {}),
        ("Parallel, several per step", PARALLEL,
         {"system": SYSTEM_SEVERAL, "several": True}),
        ("Parallel, several, naive parser", PARALLEL,
         {"system": SYSTEM_SEVERAL, "several": True, "read": read_naive}),
        ("Prices", PRICE_QUESTIONS, {}),
        ("Prices, forcing a tool", PRICE_QUESTIONS, {"force": True})]:
    results[name] = solve(tasks, **options)
    print(f"{name:40s}", summary(results[name]))
print(count_calls(ANSWERS))

## 6. The small model

Qwen3-1.7B with the same program. First, the memory of the large model is freed.

In [ ]:
del model
torch.cuda.empty_cache()
tok, model = load(CPU_MODEL)
reset()
print(agent("How much does the FVRCP vaccine cost?")[0])
reset()
print(agent("How much does the FVRCP vaccine cost?", force=True)[0])

In [ ]:
for name, options in [("Prices", {}),
                      ("Prices, forcing a tool", {"force": True})]:
    print(f"{name:40s}", summary(solve(PRICE_QUESTIONS, **options)))

## Exercises

1. Explain in your own words what the chat template does. Where do the schemas of the tools end up? And the results?

2. Print the text that `template` generates for a task with only two tools and find the fixed instructions the template adds. How many tokens do they take? Do they change if you change the tools?

3. Check with the tokenizer that `<tool_call>` is a single token. Do the same with `<tool_response>` and with `<tools>`. What is the advantage of their being single tokens?

4. Write by hand five hard answers for the parsers: two calls stuck together with no line break, a call with text after it, a JSON with single quotes, a call with no closing tag and an answer that mentions the tag in a sentence. What does each parser return?

5. Run the task about the debt of three clients with the naive parser and `show=True`. In which step does the model ask for the calls that were lost?

6. Write two tasks in which asking for several calls at once would be a mistake, because the second one needs the result of the first. Does the agent with `SYSTEM_SEVERAL` ask for them at once?

7. Remove the "Thought:" line from `SYSTEM_ONE` and try Qwen3-1.7B, without forcing, with the twelve tasks of chapter 1. Does it call the tools now?

8. Force a specific tool, not just a call: the prefix can include the start of the JSON with the name. Use it so that the price questions always start with `price`. Does anything change with the small model?

9. Try the small model forcing every step, not just the first. What happens when it already has the answer and is forced to call a tool?

10. **Project.** Add to the clinic a tool `services()` with no arguments, which returns the list of services with their price, and remove `price`. Measure the price questions with both models, with and without forcing, and compare the tokens of their results with those of `price`. Which design do you prefer?

## References

Models [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507) and [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B). Transformers documentation on [tools and chat templates](https://huggingface.co/docs/transformers/main/en/chat_extras) and Qwen's on [function calling](https://qwen.readthedocs.io/en/latest/framework/function_call.html). Northside Veterinary Clinic and its data are fictional. Original code and experiments.